# Imports and Setup


In [1]:

import pandas as pd
import numpy as np
from sodapy import Socrata

APP_KEY = "t2GPRE1GXlktxvqOIFMelPIvT"

version = "5"

client = Socrata(
    "data.cityofnewyork.us",
    APP_KEY,
    timeout=90
)

RELEASE_NAMES = ["prelim", "exec", "adopt"]

# Expense Budget


## API string


https://data.cityofnewyork.us/api/v3/views/mwzb-yiwb/query.json


In [2]:
EXPENSE_DATASET_ID = "mwzb-yiwb"

## Check Fiscal Year


In [3]:
# 1. Fetch the 2 most recent unique fiscal years present in the dataset
year_query = "SELECT fiscal_year GROUP BY fiscal_year ORDER BY fiscal_year DESC LIMIT 2"
year_json = client.get(EXPENSE_DATASET_ID, query=year_query)

In [4]:
# Extract the years into a list, e.g., ['2026', '2025']
planned_fy = [row['fiscal_year'] for row in year_json][0]

planned_fy_name = f"FY{int(planned_fy) % 100}"

print(planned_fy)
print(planned_fy_name)

print()

current_fy = int(planned_fy) - 1
current_fy_name = f"FY{int(current_fy) % 100}"

print(current_fy)
print(current_fy_name)

2027
FY27

2026
FY26


In [5]:
str_sum_current_adopted_amount = f"{current_fy_name}_adopted_amount"
str_sum_current_modified_amount = f"{current_fy_name}_current_modified_amount"
str_sum_financial_plan_amount = f"{planned_fy_name}_financial_plan_amount"

str_sum_current_adopted_position = f"{current_fy_name}_adopted_position"
str_sum_current_modified_position = f"{current_fy_name}_current_modified_position"
str_sum_financial_plan_position = f"{planned_fy_name}_financial_plan_position"

str_sum_current_adopted_number_of_contracts = f"{current_fy_name}_adopted_number_of_contracts"
str_sum_current_modified_number_of_contracts = f"{current_fy_name}_current_modified_number_of_contracts"
str_sum_financial_plan_number_of_contracts = f"{planned_fy_name}_financial_plan_number_of_contracts"


# str_sum_current_adopted_amount = "adopted_budget_amount"
# str_sum_current_modified_amount = "current_modified_budget_amount"
# str_sum_financial_plan_amount = "financial_plan_amount"

# str_sum_current_adopted_position = "adopted_budget_position"
# str_sum_current_modified_position = "current_modified_budget_position"
# str_sum_financial_plan_position = "financial_plan_position"


where_string = f"fiscal_year IN ({planned_fy})"

print(f"where_string:\n{where_string}\n")

orderby_arr = [
    "agency_number", # 2: 
    "unit_appropriation_number", # 4: 
    "responsibility_center_code", # 23: 
    "budget_code_number", # 6: 
    "object_class_number", # 8: 
    "object_code", # 10: 
    "publication_date DESC"
]

orderby_string = ", ".join(orderby_arr)

print(f"orderby_string:\n{orderby_string}\n")

where_string:
fiscal_year IN (2027)

orderby_string:
agency_number, unit_appropriation_number, responsibility_center_code, budget_code_number, object_class_number, object_code, publication_date DESC



## Query Data if necessary


In [6]:
data = []
offset = 0

limit = 1000

try:
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")
except:
    print(f"Querying for fiscal year {planned_fy}_v{version}:\n")
    while True:
        print(f"Fetching rows starting at offset: {offset}")

        temp = client.get(
            dataset_identifier=EXPENSE_DATASET_ID,
            # select=select_string,
            where=where_string,
            # group=groupby_string,
            order=orderby_string,
            # order=":id",
            limit=limit,
            offset=offset
        )
        
        if not temp:
            break
        
        # print(temp[0])
        
        data.extend(temp)
        
        offset += limit
        
        # break
            
    print(f"End of query. Successfully fetched {len(data)} total rows.")
    
    len(data)
    
    # Convert to pandas DataFrame
    df = pd.DataFrame.from_records(data)
    # df.drop(columns=["financial_plan_savings_flag"], inplace=True)
    
    df.to_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv", index=False)
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")

/var/folders/kq/xpb5g0qn23g4s4hvlvhwdgn80000gn/T/ipykernel_15518/944054463.py:7: DtypeWarning: Columns (0: agency_number, 1: responsibility_center_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")


In [7]:
len(df)

65000

## Sort and reorder


In [8]:
for i, col in enumerate(df.columns):
    print(f"{i}: '{col}',")

0: 'publication_date',
1: 'fiscal_year',
2: 'agency_number',
3: 'agency_name',
4: 'unit_appropriation_number',
5: 'unit_appropriation_name',
6: 'budget_code_number',
7: 'budget_code_name',
8: 'object_class_number',
9: 'object_class_name',
10: 'object_code',
11: 'object_code_name',
12: 'responsibility_center_code',
13: 'responsibility_center_name',
14: 'personal_service_other_than_personal_service_indicator',
15: 'financial_plan_savings_flag',
16: 'adopted_budget_amount',
17: 'current_modified_budget_amount',
18: 'financial_plan_amount',
19: 'adopted_budget_position',
20: 'current_modified_budget_position',
21: 'financial_plan_position',
22: 'adopted_budget_number_of_contracts',
23: 'current_modified_budget_number_of_contracts',
24: 'financial_plan_number_of_contracts',
25: 'intra_city_purchase_code',


In [9]:
rename_cols = {
    "adopted_budget_amount": f"{str_sum_current_adopted_amount}",
    "current_modified_budget_amount": f"{str_sum_current_modified_amount}",
    "financial_plan_amount": f"{str_sum_financial_plan_amount}",
    "adopted_budget_position": f"{str_sum_current_adopted_position}",
    "current_modified_budget_position": f"{str_sum_current_modified_position}",
    "financial_plan_position": f"{str_sum_financial_plan_position}",
    "adopted_budget_number_of_contracts": f"{str_sum_current_adopted_number_of_contracts}",
    "current_modified_budget_number_of_contracts": f"{str_sum_current_modified_number_of_contracts}",
    "financial_plan_number_of_contracts": f"{str_sum_financial_plan_number_of_contracts}",
    }

df = df.rename(columns=rename_cols)

In [10]:
# numeric_cols = [
#     "adopted_budget_amount",
#     "current_modified_budget_amount",
#     "financial_plan_amount",
    
#     "adopted_budget_position",
#     "current_modified_budget_position",
#     "financial_plan_position",
    
#     "adopted_budget_number_of_contracts",
#     "current_modified_budget_number_of_contracts",
#     "financial_plan_number_of_contracts"
# ]

numeric_cols = [
str_sum_current_adopted_amount,
str_sum_current_modified_amount,
str_sum_financial_plan_amount,

str_sum_current_adopted_position,
str_sum_current_modified_position,
str_sum_financial_plan_position,

str_sum_current_adopted_number_of_contracts,
str_sum_current_modified_number_of_contracts,
str_sum_financial_plan_number_of_contracts,
]

target_categorical_cols = [
    # "publication_date",
    # "fiscal_year",
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
]

master_categorical_cols = [
    "publication_date",
    # "fiscal_year",
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
]

all_categorical_cols = [
    "publication_date",
    "fiscal_year",
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    "financial_plan_savings_flag",
    "intra_city_purchase_code",
    "personal_service_other_than_personal_service_indicator",
]

In [11]:
# for c in df.columns:
#     print(c)

In [12]:
df = df.sort_values(
    [
        "agency_number",
        "unit_appropriation_number",
        "responsibility_center_code",
        "budget_code_number",
        "object_class_number",
        "object_code",
        "publication_date"
        ],
    ascending=[True, True, True, True, True, True, False]
    ).reset_index(drop=True)
df

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts,intra_city_purchase_code
0,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0229,Counsel to the Mayor,1,FULL TIME SALARIED,...,1789231,1789231,1789231,8,8,8,0,0,0,NaN
1,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0229,Counsel to the Mayor,1,FULL TIME SALARIED,...,1789231,1789231,1789231,8,8,8,0,0,0,NaN
2,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0230,Mayor's Judiciary Committee,1,FULL TIME SALARIED,...,88404,237404,88404,1,1,1,0,0,0,NaN
3,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0230,Mayor's Judiciary Committee,1,FULL TIME SALARIED,...,88404,168404,88404,1,1,1,0,0,0,NaN
4,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0245,Comm to Combat Domestic Violence,1,FULL TIME SALARIED,...,1870336,1870336,1870336,12,12,12,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64995,20260217,2027,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,P010,Fleet Reduction,40,OTHER SERVICES AND CHARGES,...,0,0,-1060000000,0,0,0,0,0,0,NaN
64996,20260512,2027,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,90,OTPS HOLDING CODES,...,0,0,-400000000,0,0,0,0,0,0,NaN
64997,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,...,0,0,37689973,0,0,0,0,0,0,NaN
64998,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P003,Tier IV RECs,40,OTHER SERVICES AND CHARGES,...,0,0,75000000,0,0,0,0,0,0,NaN


In [13]:
# Reorder columns
df = df[
    [
    "publication_date", # 0: 
    "fiscal_year", # 1: 
    "agency_number", # 2: 
    "agency_name", # 3: 
    "unit_appropriation_number", # 4: 
    "unit_appropriation_name", # 5: 
    "responsibility_center_code", # 23: 
    "responsibility_center_name", # 24: 
    "budget_code_number", # 6: 
    "budget_code_name", # 7: 
    "object_class_number", # 8: 
    "object_class_name", # 9: 
    "object_code", # 10: 
    "object_code_name", # 11: 
    "intra_city_purchase_code", # 25: 
    "personal_service_other_than_personal_service_indicator", # 12: 
    "financial_plan_savings_flag", # 13: 
    *numeric_cols
    # "adopted_budget_amount", # 14: 
    # "current_modified_budget_amount", # 15: 
    # "financial_plan_amount", # 16: 
    # "adopted_budget_position", # 17: 
    # "current_modified_budget_position", # 18: 
    # "financial_plan_position", # 19: 
    # "adopted_budget_number_of_contracts", # 20: 
    # "current_modified_budget_number_of_contracts", # 21: 
    # "financial_plan_number_of_contracts", # 22: 
    ]
]

df.head()

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,...,financial_plan_savings_flag,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts
0,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,Counsel to the Mayor,...,N,1789231,1789231,1789231,8,8,8,0,0,0
1,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,Counsel to the Mayor,...,N,1789231,1789231,1789231,8,8,8,0,0,0
2,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,Mayor's Judiciary Committee,...,N,88404,237404,88404,1,1,1,0,0,0
3,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,Mayor's Judiciary Committee,...,N,88404,168404,88404,1,1,1,0,0,0
4,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0245,Comm to Combat Domestic Violence,...,N,1870336,1870336,1870336,12,12,12,0,0,0


## Clean


In [14]:
print(master_categorical_cols)

df_pivot = df[master_categorical_cols+numeric_cols].groupby(master_categorical_cols, dropna=False).sum().reset_index()

# df_pivot = df_pivot[df_pivot['publication_date'] == 20260512]

df_pivot

['publication_date', 'agency_number', 'agency_name', 'unit_appropriation_number', 'unit_appropriation_name', 'responsibility_center_code', 'responsibility_center_name', 'budget_code_number', 'budget_code_name', 'object_class_number', 'object_class_name', 'object_code', 'object_code_name']


,publication_date,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,...,object_code_name,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts
0,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,Counsel to the Mayor,1,...,FULL YEAR POSITIONS,1789231,1789231,1789231,8,8,8,0,0,0
1,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,Mayor's Judiciary Committee,1,...,FULL YEAR POSITIONS,88404,168404,88404,1,1,1,0,0,0
2,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0245,Comm to Combat Domestic Violence,1,...,FULL YEAR POSITIONS,1870336,1870336,1870336,12,12,12,0,0,0
3,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0250,Office of Immigrant Affairs,1,...,FULL YEAR POSITIONS,778962,778962,778962,5,5,5,0,0,0
4,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0008,D/M FOR HUMAN SVC,0217,D/M FOR HEALTH & HUMAN SERVICES,1,...,FULL YEAR POSITIONS,1348827,1348827,1348827,7,7,7,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64516,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,...,HEAT LIGHT & POWER,9027,9027,13075,0,0,0,0,0,0
64517,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,...,OTHER EXPENSES - GENERAL,8688,7718,8688,0,0,0,0,0,0
64518,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,60,...,CONTRACTUAL SERVICES GENERAL,0,9600,0,0,0,0,0,1,0
64519,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,NaN,NaN,P002,FINANCIAL PLAN SAVINGS,40,...,OTHER EXPENSES - GENERAL,22363,97,-2637,0,0,0,0,0,0


In [15]:
for cat_col in master_categorical_cols:
    if cat_col in df_pivot.columns:
        df_pivot[cat_col] = df_pivot[cat_col].apply(lambda x: str.upper(x) if type(x) == str else x)

In [16]:
df_pivot[master_categorical_cols] = df_pivot[master_categorical_cols].fillna("(blank)")
df_pivot[numeric_cols] = df_pivot[numeric_cols].fillna(0)
df_pivot

,publication_date,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,...,object_code_name,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts
0,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,COUNSEL TO THE MAYOR,1,...,FULL YEAR POSITIONS,1789231,1789231,1789231,8,8,8,0,0,0
1,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,MAYOR'S JUDICIARY COMMITTEE,1,...,FULL YEAR POSITIONS,88404,168404,88404,1,1,1,0,0,0
2,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0245,COMM TO COMBAT DOMESTIC VIOLENCE,1,...,FULL YEAR POSITIONS,1870336,1870336,1870336,12,12,12,0,0,0
3,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0250,OFFICE OF IMMIGRANT AFFAIRS,1,...,FULL YEAR POSITIONS,778962,778962,778962,5,5,5,0,0,0
4,20260217,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0008,D/M FOR HUMAN SVC,0217,D/M FOR HEALTH & HUMAN SERVICES,1,...,FULL YEAR POSITIONS,1348827,1348827,1348827,7,7,7,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64516,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,...,HEAT LIGHT & POWER,9027,9027,13075,0,0,0,0,0,0
64517,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,...,OTHER EXPENSES - GENERAL,8688,7718,8688,0,0,0,0,0,0
64518,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,60,...,CONTRACTUAL SERVICES GENERAL,0,9600,0,0,0,0,0,1,0
64519,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,40,...,OTHER EXPENSES - GENERAL,22363,97,-2637,0,0,0,0,0,0


## Track releases


In [17]:
pub_dates = df_pivot["publication_date"].sort_values(ascending=True).unique().tolist()[:]

num_pubs_this_year = len(pub_dates)

print(f"{num_pubs_this_year} pub_dates in FY {planned_fy}: {pub_dates}")

2 pub_dates in FY 2027: [20260217, 20260512]


## Set up base DataFrame and map function


In [18]:
base_df = df_pivot[target_categorical_cols].drop_duplicates().reset_index(drop=True)
len(base_df)

33197

## Loop through each release


In [19]:
print(pub_dates)

print(planned_fy)

for i, pub_date in enumerate(pub_dates):
    release_name = RELEASE_NAMES[i]
    
    ith_release_df = df[df['publication_date'] == pub_date]
    
    print(i, release_name)

[20260217, 20260512]
2027
0 prelim
1 exec


## Collapse Releases


In [20]:
collapsed_df = base_df.copy()

new_numeric_cols = []

start_of_numerical_cols = len(target_categorical_cols)

# collapsed_df.head()

for i, pub_date in enumerate(pub_dates):
    print(f"[{i}/{num_pubs_this_year}] pub_date -- {pub_date} ({RELEASE_NAMES[i]}):")
    
    ith_release_df = df_pivot[df_pivot["publication_date"] == pub_date][target_categorical_cols + numeric_cols].copy()
    
    print(len(ith_release_df))
    
    # print(i)
    
    # print(ith_release_df.columns[len(categorical_cols):])
    
    if i <= 2:
        for col in [str_sum_financial_plan_amount, str_sum_financial_plan_position, str_sum_financial_plan_number_of_contracts]:
            # print(col)
            if col in ith_release_df.columns:
                ith_release_df = ith_release_df.rename(
                    columns={
                        f"{col}":f"{col}_{RELEASE_NAMES[i]}",
                        }
                )
            # print()
            # print(ith_release_df.columns)
    
        if i < num_pubs_this_year - 1:
            ith_release_df = ith_release_df.drop(columns=[
                # f"adopted_budget_amount",
                # f"current_modified_budget_amount",
                str_sum_current_adopted_amount,
                str_sum_current_modified_amount,
                
                
                # f"adopted_budget_position",
                # f"current_modified_budget_position",
                str_sum_current_adopted_position,
                str_sum_current_modified_position,
                
                # f"adopted_budget_number_of_contracts",
                # f"current_modified_budget_number_of_contracts",
                str_sum_current_adopted_number_of_contracts,
                str_sum_current_modified_number_of_contracts,
                ])
            
        print(f"Adding new numeric columns:{ith_release_df.columns[start_of_numerical_cols:]}\n")
        new_numeric_cols.extend(ith_release_df.columns[start_of_numerical_cols:])
    else:
        raise Exception(f"Bad indexing, i:{i} >= num_pubs_this_year or i:{i} < 0")
    
    # print(ith_release_df.columns)
    print()
    
    collapsed_df = collapsed_df.merge(right=ith_release_df,on=target_categorical_cols, how='left')
    
    # break

# collapsed_df = collapsed_df.reset_index(drop=True)


print(len(base_df))
print(len(collapsed_df))


collapsed_df.columns

[0/2] pub_date -- 20260217 (prelim):
31667
Adding new numeric columns:Index(['FY27_financial_plan_amount_prelim',
       'FY27_financial_plan_position_prelim',
       'FY27_financial_plan_number_of_contracts_prelim'],
      dtype='str')


[1/2] pub_date -- 20260512 (exec):
32854
Adding new numeric columns:Index(['FY26_adopted_amount', 'FY26_current_modified_amount',
       'FY27_financial_plan_amount_exec', 'FY26_adopted_position',
       'FY26_current_modified_position', 'FY27_financial_plan_position_exec',
       'FY26_adopted_number_of_contracts',
       'FY26_current_modified_number_of_contracts',
       'FY27_financial_plan_number_of_contracts_exec'],
      dtype='str')


33197
33197


Index(['agency_number', 'agency_name', 'unit_appropriation_number',
       'unit_appropriation_name', 'responsibility_center_code',
       'responsibility_center_name', 'budget_code_number', 'budget_code_name',
       'object_class_number', 'object_class_name', 'object_code',
       'object_code_name', 'FY27_financial_plan_amount_prelim',
       'FY27_financial_plan_position_prelim',
       'FY27_financial_plan_number_of_contracts_prelim', 'FY26_adopted_amount',
       'FY26_current_modified_amount', 'FY27_financial_plan_amount_exec',
       'FY26_adopted_position', 'FY26_current_modified_position',
       'FY27_financial_plan_position_exec', 'FY26_adopted_number_of_contracts',
       'FY26_current_modified_number_of_contracts',
       'FY27_financial_plan_number_of_contracts_exec'],
      dtype='str')

In [21]:
new_numeric_cols

['FY27_financial_plan_amount_prelim',
 'FY27_financial_plan_position_prelim',
 'FY27_financial_plan_number_of_contracts_prelim',
 'FY26_adopted_amount',
 'FY26_current_modified_amount',
 'FY27_financial_plan_amount_exec',
 'FY26_adopted_position',
 'FY26_current_modified_position',
 'FY27_financial_plan_position_exec',
 'FY26_adopted_number_of_contracts',
 'FY26_current_modified_number_of_contracts',
 'FY27_financial_plan_number_of_contracts_exec']

In [22]:
collapsed_df

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,FY27_financial_plan_number_of_contracts_prelim,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,COUNSEL TO THE MAYOR,1,FULL TIME SALARIED,...,0.0,1789231.0,1789231.0,1789231.0,8.0,8.0,8.0,0.0,0.0,0.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,MAYOR'S JUDICIARY COMMITTEE,1,FULL TIME SALARIED,...,0.0,88404.0,237404.0,88404.0,1.0,1.0,1.0,0.0,0.0,0.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0245,COMM TO COMBAT DOMESTIC VIOLENCE,1,FULL TIME SALARIED,...,0.0,1870336.0,1870336.0,1870336.0,12.0,12.0,12.0,0.0,0.0,0.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0250,OFFICE OF IMMIGRANT AFFAIRS,1,FULL TIME SALARIED,...,0.0,778962.0,778962.0,778962.0,5.0,5.0,5.0,0.0,0.0,0.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0008,D/M FOR HUMAN SVC,0217,D/M FOR HEALTH & HUMAN SERVICES,1,FULL TIME SALARIED,...,0.0,1348827.0,1348827.0,1348827.0,7.0,7.0,7.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33192,905,DISTRICT ATTORNEY RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),0611,SPANTON OPERATING EXPENSES C00222GM,40,OTHER SERVICES AND CHARGES,...,NaN,0.0,46090.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33193,905,DISTRICT ATTORNEY RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),0611,SPANTON OPERATING EXPENSES C00222GM,40,OTHER SERVICES AND CHARGES,...,NaN,0.0,27449.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33194,906,OFFICE OF PROSECUTION SPEC NARCO,2,OTHER THAN PERSONAL SERVICES,1.0,OFFICE OF SPECIAL NAR. PROS.,0101,OFFICE OF NAR PROSECUTOR,60,CONTRACTUAL SERVICES,...,NaN,0.0,64239.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
33195,941,PUBLIC ADMINISTRATOR-NEW YORK COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-NY,1000,DEPT OPER N Y COUNTY DIVISION,40,OTHER SERVICES AND CHARGES,...,NaN,0.0,17555.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
collapsed_df[collapsed_df.duplicated(keep=False)]

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,FY27_financial_plan_number_of_contracts_prelim,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts_exec


## Object Code Level


In [24]:
Object_Code_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    # "financial_plan_savings_flag",
]

main_cols = [
    # 'adopted_budget_amount',
    # 'current_modified_budget_amount',
    f'{str_sum_current_adopted_amount}',
    f'{str_sum_current_modified_amount}',
    f'{str_sum_financial_plan_amount}_{RELEASE_NAMES[i]}',
]

object_code_collapsed_df = collapsed_df.groupby(Object_Code_cols,dropna=False).sum().reset_index()

object_code_collapsed_df = object_code_collapsed_df[Object_Code_cols + new_numeric_cols]

object_code_collapsed_df[Object_Code_cols + main_cols]


,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,object_code,object_code_name,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1158351.0,1158351.0,1158351.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1137014.0,1165014.0,1165014.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0277,SENIOR ADVISOR TO THE MAYOR,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,3406413.0,3406413.0,3406413.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0277,SENIOR ADVISOR TO THE MAYOR,3,UNSALARIED,031,UNSALARIED,85703.0,85703.0,85703.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0277,SENIOR ADVISOR TO THE MAYOR,5,AMOUNTS TO BE SCHEDULED,051,SALARY ADJUSTMENTS,9587.0,9587.0,9587.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33192,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(blank),(blank),P010,FLEET REDUCTION,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,0.0,0.0,0.0
33193,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,90,OTPS HOLDING CODES,999,OTPS HOLDING CODE,0.0,0.0,-400000000.0
33194,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,0.0,0.0,0.0
33195,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P003,TIER IV RECS,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,0.0,0.0,0.0


In [25]:
# object_code_collapsed_df[object_code_collapsed_df.duplicated(keep=False)]

## Object Class Level


In [26]:
Object_Class_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

object_class_collapsed_df = collapsed_df.groupby(Object_Class_cols,dropna=False).sum().reset_index()

object_class_collapsed_df = object_class_collapsed_df[Object_Class_cols + new_numeric_cols]

object_class_collapsed_df[Object_Class_cols + main_cols]


,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,1158351.0,1158351.0,1158351.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,1137014.0,1165014.0,1165014.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0277,SENIOR ADVISOR TO THE MAYOR,1,FULL TIME SALARIED,3406413.0,3406413.0,3406413.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0277,SENIOR ADVISOR TO THE MAYOR,3,UNSALARIED,85703.0,85703.0,85703.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0277,SENIOR ADVISOR TO THE MAYOR,5,AMOUNTS TO BE SCHEDULED,9587.0,9587.0,9587.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16532,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(blank),(blank),P010,FLEET REDUCTION,40,OTHER SERVICES AND CHARGES,0.0,0.0,0.0
16533,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,90,OTPS HOLDING CODES,0.0,0.0,-400000000.0
16534,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,0.0,0.0,0.0
16535,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P003,TIER IV RECS,40,OTHER SERVICES AND CHARGES,0.0,0.0,0.0


In [27]:
# object_code_collapsed_df[object_code_collapsed_df.duplicated(keep=False)]

## Budget Code Level


In [28]:
Budget_Code_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

budget_code_collapsed_df = collapsed_df[Budget_Code_cols + new_numeric_cols].groupby(Budget_Code_cols,dropna=False).sum().reset_index()

# budget_code_collapsed_df = budget_code_collapsed_df[Budget_Code_cols + new_numeric_cols]

# budget_code_collapsed_df[Budget_Code_cols + main_cols]

budget_code_collapsed_df

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY27_financial_plan_number_of_contracts_prelim,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1.158351e+06,7.0,0.0,1158351.0,1158351.0,1158351.0,7.0,7.0,7.0,0.0,0.0,0.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0264,NYC SERVICE OFFICE,1.165014e+06,10.0,0.0,1137014.0,1165014.0,1165014.0,10.0,10.0,10.0,0.0,0.0,0.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0277,SENIOR ADVISOR TO THE MAYOR,3.501703e+06,24.0,0.0,3501703.0,3501703.0,3501703.0,24.0,24.0,24.0,0.0,0.0,0.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),0332,NYC TOURISM GRANT,0.000000e+00,0.0,0.0,81626.0,81626.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),P020,FINANCIAL PLAN SAVINGS,0.000000e+00,40.0,0.0,0.0,0.0,6286117.0,0.0,40.0,44.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8258,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(blank),(blank),P010,FLEET REDUCTION,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8259,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
8260,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,3.768997e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8261,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),P003,TIER IV RECS,7.500000e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
# budget_code_collapsed_df[budget_code_collapsed_df.duplicated(keep=False)]

## Responsibility Center Level


In [30]:
RC_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

RC_collapsed_df = collapsed_df[RC_cols + new_numeric_cols].groupby(RC_cols,dropna=False).sum().reset_index()

# RC_collapsed_df = RC_collapsed_df[RC_cols + new_numeric_cols]

RC_collapsed_df
# RC_collapsed_df[RC_cols + main_cols]
# RC_collapsed_df[RC_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][RC_cols + main_cols]

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY27_financial_plan_number_of_contracts_prelim,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(blank),(blank),5.825068e+06,81.0,0.0,5878694.0,5906694.0,12111185.0,42.0,82.0,85.0,0.0,0.0,0.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,4.526933e+06,26.0,0.0,4526933.0,4675933.0,4526933.0,26.0,26.0,26.0,0.0,0.0,0.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0008,D/M FOR HUMAN SVC,1.348827e+06,7.0,0.0,1348827.0,1348827.0,1348827.0,7.0,7.0,7.0,0.0,0.0,0.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0011,D/M FOR FINANCE AND ECO. DEV.,1.891294e+06,9.0,0.0,1891294.0,1891294.0,1891294.0,9.0,9.0,9.0,0.0,0.0,0.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0028,D/M FOR OPERATIONS,1.248473e+06,7.0,0.0,1248473.0,1248473.0,1248473.0,7.0,7.0,7.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1918,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),-2.637000e+03,0.0,0.0,22363.0,97.0,-2637.0,0.0,0.0,0.0,0.0,0.0,0.0
1919,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(blank),(blank),-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1920,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
1921,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(blank),(blank),1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
# RC_collapsed_df[RC_collapsed_df.duplicated(keep=False)]

## Unit of Appropriation Level


In [32]:
UoA_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

UoA_collapsed_df = collapsed_df[UoA_cols + new_numeric_cols].groupby(UoA_cols,dropna=False).sum().reset_index()

# UoA_collapsed_df = UoA_collapsed_df[UoA_cols + new_numeric_cols]

UoA_collapsed_df
# UoA_collapsed_df[UoA_cols + main_cols]
# UoA_collapsed_df[UoA_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][UoA_cols + main_cols]

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY27_financial_plan_number_of_contracts_prelim,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,4.046508e+07,318.0,0.0,40515979.0,40316729.0,46751202.0,319.0,319.0,322.0,0.0,0.0,0.0
1,2,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS,4.489982e+06,0.0,14.0,4237982.0,4459520.0,5031799.0,0.0,0.0,0.0,14.0,18.0,14.0
2,2,MAYORALTY,40,OFFICE OF MGMT AND BUDGET-PS,5.606719e+07,467.0,0.0,55440649.0,54732488.0,55900321.0,455.0,452.0,467.0,0.0,0.0,0.0
3,2,MAYORALTY,41,OFFICE OF MGMT AND BUDGET-OTPS,1.134907e+07,0.0,23.0,11306577.0,11537599.0,11666672.0,0.0,0.0,0.0,23.0,23.0,23.0
4,2,MAYORALTY,61,OFF OF LABOR RELATIONS-PS,1.811062e+07,188.0,0.0,16885354.0,17866722.0,18597617.0,173.0,193.0,193.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
722,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,3.105700e+04,0.0,0.0,56057.0,56057.0,35105.0,0.0,0.0,0.0,0.0,1.0,0.0
723,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
724,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
725,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
# UoA_collapsed_df[UoA_collapsed_df.duplicated(keep=False)]

## Agency Level


In [34]:
Agency_cols = [
    "agency_number",
    "agency_name",
    # "unit_appropriation_number",
    # "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

Agency_collapsed_df = collapsed_df[Agency_cols + new_numeric_cols].groupby(Agency_cols,dropna=False).sum().reset_index()

# Agency_collapsed_df = Agency_collapsed_df[Agency_cols + new_numeric_cols]

# Agency_collapsed_df[Agency_cols + main_cols]
Agency_collapsed_df

,agency_number,agency_name,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY27_financial_plan_number_of_contracts_prelim,FY26_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec,FY26_adopted_number_of_contracts,FY26_current_modified_number_of_contracts,FY27_financial_plan_number_of_contracts_exec
0,2,MAYORALTY,1.881795e+08,1311.0,65.0,196146351.0,197354723.0,199358009.0,1289.0,1306.0,1332.0,65.0,77.0,65.0
1,3,BOARD OF ELECTIONS,1.468751e+08,517.0,37.0,146875148.0,216647571.0,206972679.0,517.0,517.0,517.0,37.0,33.0,37.0
2,4,CAMPAIGN FINANCE BOARD,1.347705e+07,87.0,27.0,109481237.0,123881237.0,104093091.0,258.0,258.0,258.0,9.0,11.0,10.0
3,8,OFFICE OF THE ACTUARY,7.617044e+06,42.0,9.0,7614099.0,7677099.0,8483139.0,42.0,42.0,42.0,9.0,10.0,9.0
4,10,BOROUGH PRESIDENT - MANHATTAN,5.586632e+06,56.0,0.0,6017382.0,6194656.0,6079019.0,56.0,56.0,56.0,0.0,9.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,6.555800e+05,5.0,0.0,673602.0,673602.0,657628.0,5.0,5.0,5.0,0.0,1.0,0.0
142,99C,CITYWIDE SAVINGS INITIATIVES,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
143,99D,PRIOR YEAR PAYABLES,0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
144,99P,ENERGY ADJUSTMENT,1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Write to excel


In [35]:
def add_diff_cols(sheet_name, df):
    print(sheet_name)
    
    new_df = df.copy()
    
    
    
    collapsed_diff_df = collapsed_df.copy()

    for i in range(0,num_pubs_this_year):
        
        # "total_adopted_budget_amount",
        # "total_current_modified_budget_amount",
        # "total_adopted_budget_position",
        # "total_current_modified_budget_position"

        str_sum_current_adopted_amount
        str_sum_current_modified_amount

        str_sum_current_adopted_position
        str_sum_current_modified_position


        amount_change_prev_adopt_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
        # amount_change_percent_prev_adopt_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"

        collapsed_diff_df[amount_change_prev_adopt_name]= collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount]
        # collapsed_diff_df[amount_change_percent_prev_adopt_name]= (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount])/collapsed_df[i_amount_name]


        amount_change_prev_modified_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"
        # amount_change_percent_prev_modified_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

        collapsed_diff_df[amount_change_prev_modified_name] = collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount]
        # collapsed_diff_df[amount_change_percent_prev_modified_name] = (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount])/collapsed_df[i_amount_name]


        position_change_prev_adopt_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
        position_change_prev_modified_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

        collapsed_diff_df[position_change_prev_adopt_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_adopted_position]
        collapsed_diff_df[position_change_prev_modified_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_modified_position]

        # collapsed_diff_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-adopt"] = (collapsed_df[i_position_name] - collapsed_df["total_adopted_budget_position"])/collapsed_df[i_position_name]
        # collapsed_diff_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-modified"] = (collapsed_df[i_position_name] - collapsed_df["total_current_modified_budget_position"])/collapsed_df[i_position_name]
        
        
        for j in range(0,i):
            print(f"i={i}, j={j}")
            print(f"{RELEASE_NAMES[i]} - {RELEASE_NAMES[j]}")
            
            i_amount_name = f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[i]}"
            j_amount_name = f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[j]}"
            
            i_position_name = f"{str_sum_financial_plan_position}_{RELEASE_NAMES[i]}"
            j_position_name = f"{str_sum_financial_plan_position}_{RELEASE_NAMES[j]}"
            
            print()
            print(f"i_amount_name: {i_amount_name}")
            print(f"j_amount_name: {j_amount_name}")
            print()
            print(f"i_position_name: {i_position_name}")
            print(f"j_position_name: {j_position_name}")
            print()
                
                
            amount_change_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
            # amount_change_percent_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
            
            print(amount_change_name)
            # print(amount_change_percent_name)
                
            collapsed_diff_df[amount_change_name] = collapsed_df[i_amount_name] - collapsed_df[j_amount_name]
            # collapsed_diff_df[amount_change_percent_name] = (collapsed_df[i_amount_name] - collapsed_df[j_amount_name])/collapsed_df[i_amount_name]
            
            
            print()
            
            position_change_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
            # position_change_percent_name = f"{str_sum_financial_plan_position}_%change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
            
            print(position_change_name)
            # print(position_change_percent_name)
            
            collapsed_diff_df[position_change_name] = collapsed_df[i_position_name] - collapsed_df[j_position_name]
            # collapsed_diff_df[position_change_percent_name] = (collapsed_df[i_position_name] - collapsed_df[j_position_name])/collapsed_df[i_position_name]
            
            print("="*50)
    
    
    
    
    return new_df

In [ ]:
from openpyxl.utils import get_column_letter

output_filename = f"current_api_{planned_fy}_v{version}.xlsx"

# 1. Group your sheets into a dictionary to easily loop through them
sheets_to_write = {
    "Raw": df_pivot,
    "Collapsed": collapsed_df,
    "Object Code": object_code_collapsed_df,
    "Object Class": object_class_collapsed_df,
    "Budget Code": budget_code_collapsed_df,
    "Responsibility Center": RC_collapsed_df,
    "Unit of Appropriation": UoA_collapsed_df,
    "Agency": Agency_collapsed_df
}

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    for sheet_name, df in sheets_to_write.items():
        
        # df = add_diff_cols(sheet_name, df)
        
        # A. Write the dataframe to the current sheet
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        if sheet_name != "Raw":
            
            # B. Access the active openpyxl worksheet object pandas just generated
            worksheet = writer.sheets[sheet_name]
            
            # C. Loop through the columns of the DataFrame
            # (enumerate start=1 because Excel columns are 1-indexed: A=1, B=2...)
            for col_idx, col_name in enumerate(df.columns, start=1):
                
                # Check if this column is one of your targets
                if col_name in new_numeric_cols:
                    
                    # Assign formatting style: Currency vs Headcount Positions
                    if "position" in col_name.lower() or "contract" in col_name.lower():
                        num_format = '#,##0'    # E.g., 2,450
                    else:
                        num_format = '$#,##0'   # E.g., $1,250,000 (No decimals for clean budgets)
                    
                    # Apply the format to every cell in this column, skipping the header (row 1)
                    for row in range(2, worksheet.max_row + 1):
                        worksheet.cell(row=row, column=col_idx).number_format = num_format
                    
                    # BONUS: Set a wider column width so Excel doesn't clip numbers into '###'
                    col_letter = get_column_letter(col_idx)
                    worksheet.column_dimensions[col_letter].width = 22

print(f"Successfully saved and formatted multiple sheets to {output_filename}!")

Successfully saved and formatted multiple sheets to current_api_2027_v5.xlsx!


In [ ]:
# from openpyxl.utils import get_column_letter

# output_filename = f"current_api_{planned_fy}_v{version}.xlsx"

# # 1. Group your sheets into a dictionary to easily loop through them
# sheets_to_write = {
#     "Raw": df_pivot,
#     "Collapsed": collapsed_df,
#     "Object Code": object_code_collapsed_df,
#     "Object Class": object_class_collapsed_df,
#     "Budget Code": budget_code_collapsed_df,
#     "Responsibility Center": RC_collapsed_df,
#     "Unit of Appropriation": UoA_collapsed_df,
#     "Agency": Agency_collapsed_df
# }

# with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
#     for sheet_name, df in sheets_to_write.items():
#         # A. Write the dataframe to the current sheet
#         df.to_excel(writer, sheet_name=sheet_name, index=False)
        
#         if sheet_name != "Raw":
            
#             # B. Access the active openpyxl worksheet object pandas just generated
#             worksheet = writer.sheets[sheet_name]
            
#             # C. Loop through the columns of the DataFrame
#             # (enumerate start=1 because Excel columns are 1-indexed: A=1, B=2...)
#             for col_idx, col_name in enumerate(df.columns, start=1):
                
#                 # Check if this column is one of your targets
#                 if col_name in new_numeric_cols:
                    
#                     # Assign formatting style: Currency vs Headcount Positions
#                     if "position" in col_name.lower() or "contract" in col_name.lower():
#                         num_format = '#,##0'    # E.g., 2,450
#                     else:
#                         num_format = '$#,##0'   # E.g., $1,250,000 (No decimals for clean budgets)
                    
#                     # Apply the format to every cell in this column, skipping the header (row 1)
#                     for row in range(2, worksheet.max_row + 1):
#                         worksheet.cell(row=row, column=col_idx).number_format = num_format
                    
#                     # BONUS: Set a wider column width so Excel doesn't clip numbers into '###'
#                     col_letter = get_column_letter(col_idx)
#                     worksheet.column_dimensions[col_letter].width = 22

# print(f"Successfully saved and formatted multiple sheets to {output_filename}!")